_Supervised learning for classification_

In [ ]:
# Import packages
import pickle
import os
import pandas as pd
import numpy as np
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.figure_factory as ff
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from IPython.display import display
from sklearn.metrics import (accuracy_score, 
                             precision_score, 
                             recall_score, 
                             f1_score, 
                             confusion_matrix,
                             roc_curve,
                             auc)
# Set notebook mode to work in offline
pyo.init_notebook_mode()

In [ ]:
# Load the matrix with observation as columns and CpGs as rows
input_path = "../../output/processed_mVals.csv"
data = pd.read_csv(input_path, sep=",", index_col=0)

# Restrict to unknown samples if they are marked with the 'unsure.' prefix.
unknown_data = data.filter(regex=r'^unsure\.')
if unknown_data.shape[1] == 0:
    print("No unknown samples matching '^unsure\\.' were found in processed_mVals.csv")
else:
    print("Unknown samples detected:")
    print(unknown_data.columns)

In [ ]:
# Build labels for unknown samples. If names are not in Group.sample format,
# assign the group label 'Unknown'.
labels = pd.DataFrame(data=unknown_data.columns, columns=["label"])
if labels.empty:
    labels["Group"] = pd.Series(dtype='object')
    labels["sample"] = pd.Series(dtype='object')
else:
    split_labels = labels['label'].str.split('.', n=1, expand=True)
    if split_labels.shape[1] == 2:
        labels[['Group', 'sample']] = split_labels
    else:
        labels['Group'] = 'Unknown'
        labels['sample'] = labels['label']
display(labels)

In [ ]:
if unknown_data.empty:
    print("No unknown sample matrix available for summary statistics.")
else:
    display(unknown_data.describe())

In [ ]:
print(unknown_data.shape)
print(unknown_data.info())
# print(unknown_data.head())

In [ ]:
# Load the held-out test set and the fitted feature-selection pipeline.
with open("outputs/03_Variables.pkl", 'rb') as file:
    (X_train, X_test, y_train, y_test, kfold) = pickle.load(file)
with open("outputs/03_FeatureSelection.pkl", 'rb') as file:
    feature_selection_artifacts = pickle.load(file)

variance_selector = feature_selection_artifacts['variance_selector']
univariate_selector = feature_selection_artifacts['univariate_selector']
mi_selector = feature_selection_artifacts['mi_selector']
scaler = feature_selection_artifacts['scaler']
rfe_selector = feature_selection_artifacts['rfe_selector']
selected_columns = feature_selection_artifacts['selected_columns']

In [ ]:
if unknown_data.empty:
    X_unknown = pd.DataFrame(columns=selected_columns)
else:
    unknown_samples = unknown_data.transpose()
    raw_feature_names = list(getattr(variance_selector, 'feature_names_in_', data.index.astype(str)))
    unknown_samples = unknown_samples.reindex(columns=raw_feature_names, fill_value=0)

    X_unknown_vt = pd.DataFrame(
        variance_selector.transform(unknown_samples),
        index=unknown_samples.index,
        columns=np.array(raw_feature_names)[variance_selector.get_support(indices=True)]
    )
    X_unknown_uni = pd.DataFrame(
        univariate_selector.transform(X_unknown_vt),
        index=X_unknown_vt.index,
        columns=X_unknown_vt.columns[univariate_selector.get_support(indices=True)]
    )
    X_unknown_mi = pd.DataFrame(
        mi_selector.transform(X_unknown_uni),
        index=X_unknown_uni.index,
        columns=X_unknown_uni.columns[mi_selector.get_support(indices=True)]
    )
    X_unknown_scaled = scaler.transform(X_unknown_mi)
    X_unknown = pd.DataFrame(
        rfe_selector.transform(X_unknown_scaled),
        index=X_unknown_mi.index,
        columns=selected_columns
    )

display(X_unknown.head())

In [ ]:
model_logistic_regression = joblib.load('outputs/03-1_Logistic_Regression_final_model.joblib')
model_decision_tree = joblib.load('outputs/03-2_Decision_Tree_final_model.joblib')
model_random_forest = joblib.load('outputs/03-3_Random_Forest_final_model.joblib')
model_SVM = joblib.load('outputs/03-4_SVM_final_model.joblib')

trained_models = {"Logistic Regression": model_logistic_regression,
                  "Decision Tree": model_decision_tree,
                  "Random Forest": model_random_forest,
                  "SVM": model_SVM}

# X_test = filtered_data.T
# y_test = labels['Group']

In [ ]:
# Keep the held-out test set immutable. Build a separate matrix for application-time scoring.
X_application = X_test.copy()
y_application = y_test.copy()

if not X_unknown.empty:
    X_application = pd.concat([X_application, X_unknown], axis=0)
    y_application = pd.concat([y_application, labels.set_index('label').loc[X_unknown.index, 'Group']])

print('Held-out test samples:', X_test.shape[0])
print('Application scoring samples:', X_application.shape[0])


In [ ]:
# Score the held-out test set plus any transformed unknown samples
sample_names = X_application.index
# Create lists to store prediction scores for each model
model_names = []
prediction_scores = []

# Loop through each trained model
for model_name in trained_models:
    # Get prediction scores on the test dataset
    y_scores = trained_models[model_name].predict_proba(X_application)
    
    # Append model name and scores to lists
    model_names.append(model_name)
    prediction_scores.append(y_scores)

# Create a DataFrame to store the prediction scores for each model and each class
scores_df = pd.concat(
    [pd.DataFrame(
        scores,
        columns=[f"{model_name}_{class_name}" for class_name in trained_models[model_name].classes_],
        index=sample_names
    ) for model_name, scores in zip(model_names, prediction_scores)], axis=1)

# Transpose the DataFrame
scores_df = scores_df.T

# Display the first few rows of the transposed DataFrame
display(scores_df.head())

# Visualize the scores using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(scores_df, cmap="viridis", cbar=True, yticklabels=scores_df.index)
plt.ylabel('Model and Class')
plt.xlabel('Sample Index')
plt.title('Prediction Scores for Each Class by Model')
plt.show()

In [ ]:
scores_df